# Квантизация весов FP16/BF16 в INT4 на Triton

Каждый исходный вес занимает 16 бит. Чтобы уместить его в 4 бита, выбираем для всей строки шаг округления — масштаб. Делим вес на масштаб, округляем и ограничиваем код диапазоном от −7 до 7. Чтобы приблизительно восстановить вес, умножим его целый код на масштаб строки.

На вход подаём матрицу `W[out_features, in_features]` с чётным числом столбцов. На выходе получаем два тензора.

- `packed` — матрицу INT8 размера `[out_features, in_features // 2]`. В каждом байте хранятся два соседних кода. Первый занимает младшие 4 бита, второй — старшие.
- `scales` — по одному числу FP32 на каждую выходную строку. Для строки из нулей берём масштаб 1.

Сравним два способа выбрать шаг. В `absmax` его задаёт самый большой по модулю вес в строке. В `method="mse"` подбираем шаг по ошибке всех весов строки. Чтобы попробовать оба способа в Colab, выберите среду с GPU и запустите все ячейки.

In [1]:
import torch
import triton
import triton.language as tl
import time
import statistics
import json

assert torch.cuda.is_available(), "В Colab нужно включить GPU"
torch.manual_seed(42)
DEVICE = "cuda"
print(torch.cuda.get_device_name(0), '| PyTorch', torch.__version__, '| Triton', triton.__version__)

NVIDIA GeForce GTX 1660 Ti | PyTorch 2.10.0+cu126 | Triton 3.8.0


## Absmax в два прохода

Сначала находим максимальный модуль веса в каждой строке. Делим его на 7 и получаем шаг. Например, при максимуме 14 шаг равен 2, а код −3 означает вес −6. Во втором проходе округляем значения и сразу упаковываем соседние коды в байт. Этот вариант вызывается по умолчанию (`method="two_pass"`).

In [2]:
@triton.jit
def row_max_kernel(X, RowMax, N: tl.constexpr, BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_N)
    row_max = tl.full((), 0.0, tl.float32)
    for start in range(tl.cdiv(N, BLOCK_N)):
        j = start * BLOCK_N + cols
        x = tl.load(X + row * N + j, mask=j < N, other=0.0).to(tl.float32)
        row_max = tl.maximum(row_max, tl.max(tl.abs(x), 0))
    tl.store(RowMax + row, row_max)


@triton.jit
def quantize_pack_kernel(X, RowMax, Packed, Scales,
                         N: tl.constexpr, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    block = tl.program_id(1)
    max_val = tl.load(RowMax + row)

    scale = tl.where(max_val > 0.0, tl.div_rn(max_val, 7.0), 1.0)
    if block == 0:
        tl.store(Scales + row, scale)

    j = block * BLOCK + tl.arange(0, BLOCK)
    even = 2 * j
    odd = even + 1
    x0 = tl.load(X + row * N + even, mask=even < N, other=0.0).to(tl.float32)
    x1 = tl.load(X + row * N + odd, mask=odd < N, other=0.0).to(tl.float32)

    a0 = tl.abs(x0) * 14.0
    a1 = tl.abs(x1) * 14.0
    q0_abs = tl.full((BLOCK,), 0, tl.int32)
    q1_abs = tl.full((BLOCK,), 0, tl.int32)
    for i in tl.static_range(0, 7):
        threshold = max_val * (2 * i + 1)
        q0_abs += ((max_val > 0.0) & (a0 >= threshold)).to(tl.int32)
        q1_abs += ((max_val > 0.0) & (a1 >= threshold)).to(tl.int32)

    q0 = tl.where(x0 < 0.0, -q0_abs, q0_abs)
    q1 = tl.where(x1 < 0.0, -q1_abs, q1_abs)
    byte = ((q0 & 15) | ((q1 & 15) << 4)).to(tl.int8)
    tl.store(Packed + row * (N // 2) + j, byte, mask=j < N // 2)

### Absmax за один проход

В варианте `method="fused"` одна программа Triton берёт целую строку, находит максимум и сразу упаковывает веса. Коды и масштабы должны побайтно совпасть с результатом двух проходов. Позже проверим это на одних и тех же строках.

In [3]:
@triton.jit
def fused_quantize_pack_kernel(X, Packed, Scales,
                               N: tl.constexpr, BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    j = tl.arange(0, BLOCK_N)
    x = tl.load(X + row * N + j, mask=j < N, other=0.0).to(tl.float32)
    max_val = tl.max(tl.abs(x), 0)
    scale = tl.where(max_val > 0.0, tl.div_rn(max_val, 7.0), 1.0)
    tl.store(Scales + row, scale)

    # Квантизация выполняется над всеми элементами строки (пороговая
    # арифметика как в baseline), затем упаковываются пары соседей.
    a = tl.abs(x) * 14.0
    q_abs = tl.full((BLOCK_N,), 0, tl.int32)
    for k in tl.static_range(0, 7):
        threshold = max_val * (2 * k + 1)
        q_abs += ((max_val > 0.0) & (a >= threshold)).to(tl.int32)
    q = tl.where(x < 0.0, -q_abs, q_abs)

    # Gather по четным и нечетным позициям; по одной паре на байт.
    pairs = tl.arange(0, BLOCK_N // 2)
    q0 = tl.gather(q, 2 * pairs, axis=0)
    q1 = tl.gather(q, 2 * pairs + 1, axis=0)
    packed = ((q0 & 15) | ((q1 & 15) << 4)).to(tl.int8)
    tl.store(Packed + row * (N // 2) + pairs, packed, mask=pairs < N // 2)

## Как выбрать масштаб по ошибке весов

Если в строке есть один особенно большой вес, шаг absmax может оказаться слишком крупным для его соседей. Попробуем 18 значений шага — от половины исходного до самого исходного. При меньшем шаге небольшие веса можно округлить точнее, а самые крупные могут выйти за границы диапазона и получить крайний код −7 или 7.

Для каждого шага восстанавливаем строку и складываем квадраты разностей с исходными весами. Для одной строки минимум этой суммы даёт и минимум средней квадратичной ошибки (MSE). Считаем сумму в FP64, сохраняем выбранный масштаб в FP32. Если несколько вариантов дают одинаковую ошибку, выбираем больший шаг.

Вызов `quantize_fp16_to_int4_packed(W, method="mse")` возвращает те же `packed, scales`, которые ждёт кернель умножения из пункта 2. Для работы вне ноутбука функция есть в `quantize_int4.py`.

In [4]:
RATIOS = (1.0, 0.99, 0.975, 0.95, 0.925, 0.90, 0.875, 0.85, 0.825,
          0.80, 0.775, 0.75, 0.725, 0.70, 0.65, 0.60, 0.55, 0.50)


@triton.jit
def _abs_code(x, clip):
    # Same threshold comparisons as task 1, including ties and the zero row.
    a14 = tl.abs(x) * 14.0
    code = tl.full(x.shape, 0, tl.int32)
    for i in tl.static_range(7):
        code += (a14 >= clip * (2 * i + 1)).to(tl.int32)
    return tl.where(clip > 0.0, code, 0)


@triton.jit
def _scale(clip):
    return tl.where(clip > 0.0, tl.div_rn(clip, 7.0), 1.0)


@triton.jit
def _row_max(W, Maxima, K: tl.constexpr, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    lane = tl.arange(0, BLOCK)
    maximum = tl.full((BLOCK,), 0.0, tl.float32)
    for tile in range(tl.cdiv(K, BLOCK)):
        col = tile * BLOCK + lane
        x = tl.load(W + row * K + col, col < K, other=0.0).to(tl.float32)
        maximum = tl.maximum(maximum, tl.abs(x))
    tl.store(Maxima + row, tl.max(maximum, 0))


@triton.jit
def _score(W, Maxima, Ratios, Scores, K: tl.constexpr, COUNT: tl.constexpr, BLOCK: tl.constexpr):
    row, candidate = tl.program_id(0), tl.program_id(1)
    clip = tl.load(Maxima + row) * tl.load(Ratios + candidate)
    scale = _scale(clip)
    lane = tl.arange(0, BLOCK)
    sums = tl.full((BLOCK,), 0.0, tl.float64)
    for tile in range(tl.cdiv(K, BLOCK)):
        col = tile * BLOCK + lane
        x = tl.load(W + row * K + col, col < K, other=0.0).to(tl.float32)
        reconstructed_magnitude = _abs_code(x, clip).to(tl.float32) * scale
        error = tl.abs(x).to(tl.float64) - reconstructed_magnitude.to(tl.float64)
        sums += error * error
    tl.store(Scores + row * COUNT + candidate, tl.sum(sums, 0))


@triton.jit
def _pack_best(W, Maxima, Ratios, Scores, Packed, Scales, K: tl.constexpr,
               COUNT: tl.constexpr, PADDED_COUNT: tl.constexpr, BLOCK: tl.constexpr):
    row, tile = tl.program_id(0), tl.program_id(1)
    candidate = tl.arange(0, PADDED_COUNT)
    row_scores = Scores + row * COUNT
    scores = tl.load(row_scores + candidate, candidate < COUNT, other=float("inf"))
    # The first minimum prefers the largest retained range on equal scores.
    best = tl.argmin(scores, axis=0, tie_break_left=True)
    # Keep the canonical baseline when improvement is within FP64 reduction noise.
    best = tl.where(tl.min(scores, 0) < tl.load(row_scores) * (1.0 - 1e-12), best, 0)
    clip = tl.load(Maxima + row) * tl.load(Ratios + best)
    if tile == 0:
        tl.store(Scales + row, _scale(clip))
    j = tile * BLOCK + tl.arange(0, BLOCK)
    even = tl.load(W + row * K + 2 * j, j < K // 2, other=0.0).to(tl.float32)
    odd = tl.load(W + row * K + 2 * j + 1, j < K // 2, other=0.0).to(tl.float32)
    a, b = _abs_code(even, clip), _abs_code(odd, clip)
    a = tl.where(even < 0, -a, a)
    b = tl.where(odd < 0, -b, b)
    byte = ((a & 15) | ((b & 15) << 4)).to(tl.int8)
    tl.store(Packed + row * (K // 2) + j, byte, j < K // 2)


@torch.no_grad()
def quantize_mse(w):
    """Return int8 packed weights [rows, K/2] and one FP32 scale per row.

    Takes a CUDA FP16/BF16 matrix of finite model weights with even positive K.
    Candidate 1.0 reproduces the project's absmax quantizer. Each row selects
    a scale by SSE between the original weights and the FP32 reconstruction.
    """
    if w.ndim != 2 or w.shape[1] == 0 or w.shape[1] % 2:
        raise ValueError("Expected a matrix with positive, even width")
    if not w.is_cuda or w.dtype not in (torch.float16, torch.bfloat16):
        raise ValueError("Expected CUDA FP16 or BF16 weights")
    w = w.contiguous()
    rows, width = w.shape
    packed = torch.empty((rows, width // 2), device=w.device, dtype=torch.int8)
    scales = torch.empty(rows, device=w.device, dtype=torch.float32)
    if rows == 0:
        return packed, scales
    count = len(RATIOS)
    tile = min(512, triton.next_power_of_2(width))
    pack_tile = min(256, triton.next_power_of_2(width // 2))
    chunk = min(rows, 2048)
    # Reuse the scratch on the current CUDA stream, including for the LM head.
    with torch.cuda.device(w.device):
        ratios = torch.tensor(RATIOS, device=w.device, dtype=torch.float32)
        maxima = torch.empty(chunk, device=w.device, dtype=torch.float32)
        scores = torch.empty((chunk, count), device=w.device, dtype=torch.float64)
        for first in range(0, rows, chunk):
            n = min(rows - first, chunk)
            part = w[first:first + n]
            _row_max[(n,)](part, maxima, width, tile)
            _score[(n, count)](part, maxima, ratios, scores, width, count, tile, enable_fp_fusion=False)
            _pack_best[(n, triton.cdiv(width // 2, pack_tile))](
                part, maxima, ratios, scores, packed[first:first + n], scales[first:first + n],
                width, count, triton.next_power_of_2(count), pack_tile)
    return packed, scales


def quantize_fp16_to_int4_packed(W, method="two_pass"):
    """Симметричная per-row FP16/BF16 -> packed INT4 + FP32 scales.

    method="mse" selects the least-error scale from 18 candidates per row.
    "two_pass" (default) and "fused" use the original absmax scale.
    Контракт входа: CUDA, конечные числа, 2D, N чётное.
    Проверка конечности не выполняется внутри, чтобы не добавлять GPU sync.
    """
    assert W.is_cuda and W.ndim == 2
    assert W.dtype in (torch.float16, torch.bfloat16)
    M, N = W.shape
    assert M > 0 and N > 0 and N % 2 == 0
    W = W.contiguous()
    if method == "mse":
        return quantize_mse(W)
    if method not in ("two_pass", "fused"):
        raise ValueError("method must be 'mse', 'two_pass' or 'fused'")
    packed = torch.empty((M, N // 2), device=W.device, dtype=torch.int8)
    scales = torch.empty((M,), device=W.device, dtype=torch.float32)
    if method == "two_pass":
        row_max = torch.empty((M,), device=W.device, dtype=torch.float32)
        row_max_kernel[(M,)](W, row_max, N, BLOCK_N=1024, num_warps=4)
        B = 256
        quantize_pack_kernel[(M, triton.cdiv(N // 2, B))](
            W, row_max, packed, scales, N, B, num_warps=4
        )
    else:
        fused_quantize_pack_kernel[(M,)](
            W, packed, scales, N, BLOCK_N=triton.next_power_of_2(N), num_warps=4
        )
    return packed, scales


def decode_signed_int4(packed):
    """Возвращает signed целые [-7,7] в форме [M,N]."""
    p = packed.to(torch.int32)
    lo = p & 15
    hi = (p >> 4) & 15
    lo = torch.where(lo >= 8, lo - 16, lo)
    hi = torch.where(hi >= 8, hi - 16, hi)
    M, half = packed.shape
    q = torch.empty((M, 2 * half), device=packed.device, dtype=torch.int32)
    q[:, 0::2], q[:, 1::2] = lo, hi
    return q


def dequantize_int4_packed(packed, scales):
    return decode_signed_int4(packed).float() * scales[:, None]


def pytorch_reference(W):
    wf = W.float()
    row_max = wf.abs().amax(dim=1)
    scales = torch.where(row_max > 0, row_max / 7.0, torch.ones_like(row_max))
    abs14 = wf.abs() * 14.0
    qabs = torch.zeros_like(wf, dtype=torch.int32)
    for i in range(7):
        t = row_max[:, None] * (2 * i + 1)
        qabs += ((row_max[:, None] > 0) & (abs14 >= t)).int()
    q = torch.where(wf < 0, -qabs, qabs)
    packed = ((q[:, 0::2] & 15) | ((q[:, 1::2] & 15) << 4)).to(torch.int8)
    return packed, scales


def memory_bytes(*tensors):
    return sum(x.numel() * x.element_size() for x in tensors)

def independent_round_reference(W):
    """Математически независимый reference: half-away-from-zero, FP32.

    Из-за отличий FP32 операций от порогового метода возможны расхождения
    ровно у границ округления; их число диагностируем, но не скрываем.
    """
    wf = W.float()
    row_max = wf.abs().amax(dim=1, keepdim=True)
    scale = torch.where(row_max > 0, row_max / 7.0, torch.ones_like(row_max))
    qabs = torch.floor(wf.abs() / scale + 0.5).clamp_(0, 7).to(torch.int32)
    q = torch.where(wf < 0, -qabs, qabs)
    return q, scale.squeeze(1)

In [5]:
def check_case(W, label, method="two_pass"):
    assert torch.isfinite(W).all().item(), "Тестовые веса должны быть конечными"
    M, N = W.shape
    packed, scales = quantize_fp16_to_int4_packed(W, method=method)
    ref_packed, ref_scales = pytorch_reference(W)
    decoded = decode_signed_int4(packed)
    restored = decoded.float() * scales[:, None]
    independent_q, _ = independent_round_reference(W)
    independent_diff = (decoded != independent_q).sum().item()
    assert packed.shape == (M, N // 2) and packed.dtype == torch.int8
    assert scales.shape == (M,) and scales.dtype == torch.float32
    assert decoded.min().item() >= -7 and decoded.max().item() <= 7
    assert torch.isfinite(restored).all().item()

    mismatches = (packed != ref_packed).sum().item()
    scale_delta = (scales - ref_scales).abs().max().item()
    scales_ok = torch.allclose(scales, ref_scales, rtol=1e-6, atol=1e-7)
    err = (W.float() - restored).abs()
    # В absmax предел ошибки — половина шага. Добавляем 1e-5 на округление в FP32.
    bound_violations = (err > 0.5 * scales[:, None] + 1e-5).sum().item()
    before = memory_bytes(W)
    after = memory_bytes(packed, scales)
    ratio = before / after
    mse = err.square().mean().item()
    print(f"{label:22s} {M:5d}x{N:<5d} {str(W.dtype):14s} "
          f"mismatch={mismatches:5d} scale_diff={scale_delta:.2e} "
          f"MAE={err.mean().item():.6f} max={err.max().item():.6f} "
          f"MSE={mse:.6g} independent_diff={independent_diff} "
          f"bound_fail={bound_violations} compression={ratio:.4f}x")
    if mismatches:
        for r, c in (packed != ref_packed).nonzero()[:5].tolist():
            print(f"  byte [{r},{c}]: Triton={int(packed[r,c]) & 255:02x}, "
                  f"PyTorch={int(ref_packed[r,c]) & 255:02x}")
    assert scales_ok, "Scales diverged"
    assert mismatches == 0, "Packed bytes differ from PyTorch reference"
    assert bound_violations == 0, "Quantization error bound failed"
    assert memory_bytes(packed) * 4 == before, "Packed bytes not exactly quarter FP16/BF16"
    return {"label": label, "shape": [M, N], "dtype": str(W.dtype),
            "mismatches": mismatches, "mae": err.mean().item(),
            "max_error": err.max().item(), "mse": mse,
            "independent_q_mismatches": independent_diff,
            "compression": ratio, "method": method}

In [6]:
print("Проверка absmax по эталону PyTorch")
test_results = []
for M, N in [(1, 128), (13, 256), (2048, 2048), (8192, 2048)]:
    test_results.append(check_case(torch.randn(M, N, device=DEVICE, dtype=torch.float16), "random FP16"))
test_results.append(check_case(torch.randn(32, 256, device=DEVICE, dtype=torch.bfloat16), "random BF16"))
test_results.append(check_case(torch.zeros(3, 128, device=DEVICE, dtype=torch.float16), "all zero"))

mixed = torch.randn(5, 256, device=DEVICE, dtype=torch.float16)
mixed[0] = 0
mixed[3] = 0
test_results.append(check_case(mixed, "mixed zero rows"))

Проверка absmax по эталону PyTorch


random FP16                1x128   torch.float16  mismatch=    0 scale_diff=2.98e-08 MAE=0.089752 max=0.177734 MSE=0.0108199 independent_diff=0 bound_fail=0 compression=3.7647x
random FP16               13x256   torch.float16  mismatch=    0 scale_diff=5.96e-08 MAE=0.104574 max=0.261509 MSE=0.0150031 independent_diff=2 bound_fail=0 compression=3.8788x


random FP16             2048x2048  torch.float16  mismatch=    0 scale_diff=5.96e-08 MAE=0.129826 max=0.380301 MSE=0.0226612 independent_diff=625 bound_fail=0 compression=3.9844x
random FP16             8192x2048  torch.float16  mismatch=    0 scale_diff=5.96e-08 MAE=0.129771 max=0.377511 MSE=0.0226376 independent_diff=2466 bound_fail=0 compression=3.9844x


random BF16               32x256   torch.bfloat16 mismatch=    0 scale_diff=5.96e-08 MAE=0.109523 max=0.261719 MSE=0.0161529 independent_diff=12 bound_fail=0 compression=3.8788x
all zero                   3x128   torch.float16  mismatch=    0 scale_diff=0.00e+00 MAE=0.000000 max=0.000000 MSE=0 independent_diff=0 bound_fail=0 compression=3.7647x
mixed zero rows            5x256   torch.float16  mismatch=    0 scale_diff=2.98e-08 MAE=0.059206 max=0.212751 MSE=0.00786607 independent_diff=0 bound_fail=0 compression=3.8788x


In [7]:
# Здесь есть веса обоих знаков, постоянная строка и строка из нулей.
small = torch.tensor([[1,-2,3,-4,5,-6,7,-8],
                      [.5,.5,.5,.5,-.5,-.5,-.5,-.5],
                      [2,2,2,2,2,2,2,2],
                      [0,0,0,0,0,0,0,0]], device=DEVICE, dtype=torch.float16)
test_results.append(check_case(small, "small example"))

# При максимуме 7 шаг равен 1. Половинки должны округляться от нуля.
boundaries = torch.tensor([[7., .5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5,
                            -7., -.5, -1.5, -2.5, -3.5, -4.5, -5.5, -6.5]],
                          device=DEVICE, dtype=torch.float16)
test_results.append(check_case(boundaries, "threshold ties"))

zero_p, zero_s = quantize_fp16_to_int4_packed(torch.zeros(3,128,device=DEVICE,dtype=torch.float16))
assert torch.count_nonzero(zero_p).item() == 0
assert torch.all(zero_s == 1).item()

for bad_n in [3, 7]:
    try:
        quantize_fp16_to_int4_packed(torch.zeros((2,bad_n),device=DEVICE,dtype=torch.float16))
        raise RuntimeError("Odd-width input was unexpectedly accepted")
    except AssertionError:
        pass

small example              4x8     torch.float16  mismatch=    0 scale_diff=0.00e+00 MAE=0.071429 max=0.571429 MSE=0.0280612 independent_diff=1 bound_fail=0 compression=2.0000x


threshold ties             1x16    torch.float16  mismatch=    0 scale_diff=0.00e+00 MAE=0.437500 max=0.500000 MSE=0.21875 independent_diff=0 bound_fail=0 compression=2.6667x


In [8]:
# Первый байт должен быть 0x97. Код 7 занимает младшие биты, а код −7 — старшие.
patterns = torch.tensor([[7., -7., -1., 1., 0., 0., 3., -3.]],
                        device=DEVICE, dtype=torch.float16)
pat_p, pat_s = quantize_fp16_to_int4_packed(patterns)
assert (pat_p.to(torch.int32) & 255).tolist() == [[0x97, 0x1f, 0x00, 0xd3]]
assert decode_signed_int4(pat_p).tolist() == [[7,-7,-1,1,0,0,3,-3]]

base = torch.randn(256, 16, device=DEVICE, dtype=torch.float16)
noncontig = base[:, ::2]  # Берём каждый второй столбец, чтобы проверить вход с шагом 2.
assert not noncontig.is_contiguous()
p_nc, s_nc = quantize_fp16_to_int4_packed(noncontig)
r_nc, rs_nc = pytorch_reference(noncontig)
assert torch.equal(p_nc, r_nc)
assert torch.allclose(s_nc, rs_nc, atol=1e-7, rtol=1e-6)

print("Additional contract tests: PASS")

Additional contract tests: PASS


### Проверки на малых матрицах

Берём строки с нулями и выбросами, а также матрицы, у которых последний блок столбцов неполный. Для absmax сверяем результат с PyTorch и между двумя кернелями. Для MSE отдельно перебираем все масштабы на CPU, затем сверяем выбранный масштаб и каждый упакованный байт. Так можно заметить даже переставленные местами коды одной пары.

В проверках absmax счётчик `mismatch` показывает, сколько байтов расходится с эталоном PyTorch при тех же правилах округления. Он должен быть нулевым. Счётчик `independent_diff` сравнивает коды с отдельным расчётом через деление в FP32. На границе округления это деление иногда даёт соседний код.

In [9]:
for M, N in [(1, 2), (3, 6), (7, 130), (9, 514), (4, 2048)]:
    for dtype in (torch.float16, torch.bfloat16):
        W = torch.randn((M, N), device=DEVICE, dtype=dtype)
        test_results.append(check_case(W, f"edge {dtype}", method="two_pass"))
        p_base, s_base = quantize_fp16_to_int4_packed(W, method="two_pass")
        p_fused, s_fused = quantize_fp16_to_int4_packed(W, method="fused")
        assert torch.equal(p_base, p_fused), f"Packed mismatch on {M}x{N} {dtype}"
        assert torch.equal(s_base, s_fused), f"Scales mismatch on {M}x{N} {dtype}"

special = torch.tensor([[0, 2**-24, -2**-24, 2**-14, 0.0, 0.0],
                        [0.01, -0.02, 0.03, 0.04, 0.05, 60.0]],
                       device=DEVICE, dtype=torch.float16)
test_results.append(check_case(special, "subnormal/outlier"))
p0, s0 = quantize_fp16_to_int4_packed(special)
p1, s1 = quantize_fp16_to_int4_packed(special, method="fused")
assert torch.equal(p0, p1) and torch.equal(s0, s1)
print("Extended and fused equivalence tests: PASS")


def check_mse_selection(W, label):
    packed, scales = quantize_fp16_to_int4_packed(W, method="mse")
    ref_packed, ref_scales = [], []
    squared_error = [0.0, 0.0]
    ratios = torch.tensor(RATIOS, dtype=torch.float32)
    odd = torch.arange(1, 14, 2, dtype=torch.float32)
    for row in W.float().cpu():
        clips = ratios * row.abs().max()
        candidate_scales = torch.where(clips > 0, (clips.double() / 7).float(), 1.0)
        # Если вес попал точно на порог, его модуль округляется вверх.
        thresholds = clips[:, None] * odd[None, :]
        magnitudes = (row.abs() * 14).expand(len(ratios), -1).contiguous()
        codes = torch.searchsorted(thresholds, magnitudes, right=True)
        codes = torch.where(clips[:, None] == 0, 0, codes)
        codes = torch.where(row[None, :] < 0, -codes, codes).to(torch.int32)
        reconstructed = codes.float() * candidate_scales[:, None]
        sse = (row.double() - reconstructed.double()).square().sum(dim=1)
        index = int(sse.argmin())
        if sse[index] >= sse[0] * (1.0 - 1e-12):
            index = 0
        selected = codes[index]
        ref_packed.append(((selected[::2] & 15) | ((selected[1::2] & 15) << 4)).to(torch.int8))
        ref_scales.append(candidate_scales[index])
        squared_error[0] += sse[0].item()
        squared_error[1] += sse[index].item()

    assert packed.dtype == torch.int8 and scales.dtype == torch.float32
    assert torch.equal(packed.cpu(), torch.stack(ref_packed))
    assert torch.equal(scales.cpu(), torch.stack(ref_scales))
    old_mse, new_mse = (error / W.numel() for error in squared_error)
    assert new_mse <= old_mse
    print(f"{label}: absmax MSE {old_mse:.6g}, MSE search {new_mse:.6g}")
    return {"label": label, "method": "mse", "shape": list(W.shape),
            "absmax_mse": old_mse, "mse": new_mse}


for sample, label in [(small, "signs/constants/zero"), (boundaries, "thresholds"),
                      (special, "subnormal/outlier"), (noncontig[:8], "strided"),
                      (torch.randn(3, 514, device=DEVICE, dtype=torch.float16), "FP16 width 514"),
                      (torch.randn(2, 130, device=DEVICE, dtype=torch.bfloat16), "BF16 width 130")]:
    test_results.append(check_mse_selection(sample, label))

example = torch.tensor([[0.30, 0.30, 0.30, 0.30, 1.40, 0.00]], device=DEVICE, dtype=torch.float16)
example_result = check_mse_selection(example, "clipped scale")
assert example_result["mse"] < example_result["absmax_mse"]
test_results.append(example_result)

edge torch.float16         1x2     torch.float16  mismatch=    0 scale_diff=7.45e-09 MAE=0.020996 max=0.041992 MSE=0.000881672 independent_diff=0 bound_fail=0 compression=0.8000x


edge torch.bfloat16        1x2     torch.bfloat16 mismatch=    0 scale_diff=1.49e-08 MAE=0.021205 max=0.042411 MSE=0.000899335 independent_diff=0 bound_fail=0 compression=0.8000x


edge torch.float16         3x6     torch.float16  mismatch=    0 scale_diff=0.00e+00 MAE=0.054981 max=0.152100 MSE=0.00597062 independent_diff=0 bound_fail=0 compression=1.7143x


edge torch.bfloat16        3x6     torch.bfloat16 mismatch=    0 scale_diff=2.98e-08 MAE=0.047300 max=0.125000 MSE=0.00384875 independent_diff=0 bound_fail=0 compression=1.7143x


edge torch.float16         7x130   torch.float16  mismatch=    0 scale_diff=2.98e-08 MAE=0.105079 max=0.231027 MSE=0.0145655 independent_diff=0 bound_fail=0 compression=3.7681x


edge torch.bfloat16        7x130   torch.bfloat16 mismatch=    0 scale_diff=2.98e-08 MAE=0.098940 max=0.241769 MSE=0.0135075 independent_diff=6 bound_fail=0 compression=3.7681x


... (4 outputs left out)

thresholds: absmax MSE 0.21875, MSE search 0.0891407


subnormal/outlier: absmax MSE 0.0004583, MSE search 0.0004583
strided: absmax MSE 0.00552159, MSE search 0.00524549


FP16 width 514: absmax MSE 0.020063, MSE search 0.0131281


BF16 width 130: absmax MSE 0.0144453, MSE search 0.0117037
clipped scale: absmax MSE 0.00666574, MSE search 0.00508989


## Один крупный вес и четыре маленьких

В строке ниже четыре веса около 0,30 и один около 1,40. Шаг absmax получается примерно 0,20, а подбор по MSE даёт около 0,19. В выводе видно, как при меньшем шаге четыре маленьких веса становятся ближе к исходным, а крупный — дальше.

In [10]:
print("Исходные веса", example.tolist())
for method in ("two_pass", "mse"):
    packed, scales = quantize_fp16_to_int4_packed(example, method=method)
    print(method, "scale", scales.tolist(), "packed INT8", packed.tolist(),
          "восстановленные веса", dequantize_int4_packed(packed, scales).tolist())

Исходные веса [[0.300048828125, 0.300048828125, 0.300048828125, 0.300048828125, 1.400390625, 0.0]]
two_pass scale [0.2000558078289032] packed INT8 [[17, 17, 7]] восстановленные веса [[0.2000558078289032, 0.2000558078289032, 0.2000558078289032, 0.2000558078289032, 1.400390625, 0.0]]
mse scale [0.19005300104618073] packed INT8 [[34, 34, 7]] восстановленные веса [[0.38010600209236145, 0.38010600209236145, 0.38010600209236145, 0.38010600209236145, 1.3303710222244263, 0.0]]


### Сколько памяти занимают веса и масштабы

Для матрицы из `M` строк и `N` столбцов исходные веса занимают `2 × M × N` байт. После упаковки коды занимают `M × N / 2` байт — ровно вчетверо меньше. Ещё `4 × M` байт уходят на масштабы FP32. Итоговое сжатие составляет `4 × N / (N + 8)` раза. Чем длиннее строка, тем меньше доля памяти, которая приходится на её масштаб.

In [11]:
for M, N in [(2048, 2048), (8192, 2048), (8192, 8192)]:
    input_bytes = 2 * M * N
    packed_bytes = M * N // 2
    scales_bytes = 4 * M
    print(f"{M}x{N}: FP16={input_bytes/2**20:.2f} MiB, "
          f"packed={packed_bytes/2**20:.2f} MiB, "
          f"scales={scales_bytes/2**20:.4f} MiB, "
          f"total compression={input_bytes/(packed_bytes+scales_bytes):.4f}x")

2048x2048: FP16=8.00 MiB, packed=2.00 MiB, scales=0.0078 MiB, total compression=3.9844x
8192x2048: FP16=32.00 MiB, packed=8.00 MiB, scales=0.0312 MiB, total compression=3.9844x
8192x8192: FP16=128.00 MiB, packed=32.00 MiB, scales=0.0312 MiB, total compression=3.9961x


## Сколько времени занимает подготовка весов

Сначала создаём матрицу и прогреваем все три метода. Затем делаем шесть замеров в разном порядке и берём медиану. Время отсчитываем от вызова квантизатора до окончания работы GPU, включая выделение памяти под результат. Эти затраты возникают один раз при подготовке весов. В пункте 3 измеряется уже само умножение.

In [12]:
bench_results = []
for M, N in [(2048, 2048), (8192, 2048), (8192, 8192)]:
    W = torch.randn((M, N), device=DEVICE, dtype=torch.float16)
    methods = ("two_pass", "fused", "mse")
    for method in methods:
        quantize_fp16_to_int4_packed(W, method=method)
    torch.cuda.synchronize()
    samples = {method: [] for method in methods}
    for i in range(6):
        for method in methods[i % 3:] + methods[:i % 3]:
            start = time.perf_counter()
            packed, scales = quantize_fp16_to_int4_packed(W, method=method)
            torch.cuda.synchronize()
            samples[method].append(1000 * (time.perf_counter() - start))
            del packed, scales
    timings = {name: {"median_ms": statistics.median(times), "samples_ms": times}
               for name, times in samples.items()}
    bench_results.append({"shape": [M, N], **timings})
    print(f"{M}×{N} · медиана (мс)", {name: round(data["median_ms"], 3) for name, data in timings.items()})
    del W

2048×2048 · медиана (мс) {'two_pass': 0.471, 'fused': 0.324, 'mse': 7.563}
8192×2048 · медиана (мс) {'two_pass': 0.864, 'fused': 0.911, 'mse': 26.322}


8192×8192 · медиана (мс) {'two_pass': 1.841, 'fused': 3.712, 'mse': 84.294}


## Что сохраняется после запуска

В `int4_task1_comparison.json` записываем результаты проверок, время квантизации и формат весов. Туда же попадают модель GPU и версии PyTorch и Triton, чтобы результаты разных запусков можно было сопоставить.

In [13]:
contract = {
    "packed_dtype": "torch.int8",
    "packed_shape": "[out_features, in_features // 2]",
    "pack_axis": "in_features",
    "low_nibble": "weight[2*j]",
    "high_nibble": "weight[2*j+1]",
    "int4": "signed two's complement, used values [-7, 7]",
    "scales_dtype": "torch.float32",
    "scales_shape": "[out_features]",
    "quantization": "signed symmetric per-row with one FP32 scale, unit scale on a zero row",
    "zero_point": 0,
    "methods": {"two_pass": "absmax", "fused": "absmax", "mse": "18-candidate MSE grid"},
    "mse_clip_ratios": list(RATIOS),
    "input_precondition": "finite CUDA FP16/BF16, 2D, even in_features",
    "dequantization": "weight_hat[row, k] = signed_int4 * scales[row]",
}
results = {
    "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "triton": triton.__version__,
    "correctness": "PASS (if all notebook assertions completed)",
    "tests": test_results,
    "benchmarks": bench_results,
    "format": contract,
}
with open("int4_task1_comparison.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Результаты сохранены в int4_task1_comparison.json")
print("Интерфейс: packed, scales = quantize_fp16_to_int4_packed(W, method='mse')")

Результаты сохранены в int4_task1_comparison.json
Интерфейс: packed, scales = quantize_fp16_to_int4_packed(W, method='mse')
